# `you_01` — Extraction experiments (Ahlem)

Experiments for `oppradar_ai.ingestion` — extraction quality, dedupe thresholds, scam rules.

**Convention:** notebooks are prefixed `you_<nn>_` so Ahlem and Ikbel never collide.

In [ ]:
import json
from pathlib import Path

from oppradar_ai.ingestion.extractor import extract_batch
from oppradar_ai.ingestion.dedupe import dedupe
from oppradar_ai.ingestion.scam_check import score_opportunity

EVAL = Path("../data/eval/extraction_test_set.jsonl")
cases = [json.loads(line) for line in EVAL.read_text(encoding="utf-8").splitlines()]
len(cases)

In [ ]:
# Run the extractor on the labelled set and compare with the gold fields.
records = [c["raw"] for c in cases]
extracted = extract_batch(records)
for opp in extracted:
    print(f"{opp.title[:50]:50s} | {opp.organization[:25]:25s} | {opp.deadline}")

In [ ]:
# Scam rules on the fixtures — the synthetic scam entry should light up.
fixtures = json.loads(Path("../data/fixtures/opportunities.json").read_text(encoding="utf-8"))
from oppradar_ai.schemas import Opportunity

flagged = []
for raw in fixtures:
    opp = Opportunity.model_validate(raw)
    verdict = score_opportunity(opp)
    if verdict.label.value != "safe":
        flagged.append((opp.title, verdict.label.value, verdict.score, verdict.reasons))

flagged